In [0]:
# instaluje bibliotekę do parsowania pdf-ów
%pip install pdfplumber
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("projekt", "inspektor_budzet")
dbutils.widgets.text("dostawca", "rowkop")

projekt = dbutils.widgets.get("projekt")
dostawca = dbutils.widgets.get("dostawca")

catalog = projekt
schema = dostawca
bronze_path = f"/Volumes/{catalog}/{schema}/bronze/"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# Automatyczne wyszukiwanie pliku faktury w folderze bronze
pliki = dbutils.fs.ls(bronze_path)
pliki_faktury = [f.path for f in pliki if "faktura" in f.name.lower()]

if len(pliki_faktury) == 0:
    raise Exception(f"Nie znaleziono pliku faktury w {bronze_path}")

# Konwersja ścieżki dbfs: na /Volumes dla bibliotek Pythonowych
sciezka_pdf = pliki_faktury[0].replace("dbfs:", "")

print(f"Notebook 01b — Bronze Faktura")
print(f"Dostawca:  {dostawca}")
print(f"Plik PDF:  {sciezka_pdf}")


In [0]:
import pdfplumber

# Słownik słów kluczowych → position_id (rozszerzalny dla nowych dostawców)
keyword_map = {
    "ryczałt": "RYCZALT",
    "wykop":   "WYKOP_ROWU",
    "operator": "PRACA_OPERATORA",
    "trudny":  "DODATEK_TRUDNY_GRUNT"
}

def clean_num(s):
    return float(s.replace(" ", "").replace("\xa0", "").replace(",", ".").replace("\n", "").strip())

rows = []
# === PARSOWANIE PDF ===
with pdfplumber.open(sciezka_pdf) as pdf:
    for page in pdf.pages:
        for table in page.extract_tables():
            for row in table:
                if row[0] and row[0].strip().isdigit():
                    nazwa   = row[1] or ""
                    ilosc   = row[2] or "0"
                    jm      = row[3] or ""
                    cena    = row[4] or "0"
                    wartosc = row[5] or "0"

                    position_id = None
                    for keyword, pid in keyword_map.items():
                        if keyword in nazwa.lower():
                            position_id = pid
                            break

                    if position_id:
                        rows.append((
                            position_id,
                            clean_num(ilosc),
                            jm.strip(),
                            clean_num(cena),
                            clean_num(wartosc)
                        ))

print(f"Znaleziono {len(rows)} pozycji:")
for r in rows:
    print(r)

In [0]:
#przenoszenie danych z pdf do tabeli

df_faktura = spark.createDataFrame(
    rows,
    ["position_id", "faktura_ilosc", "faktura_jm", "faktura_cena_netto", "faktura_wartosc_netto"]
)
df_faktura.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.bronze_faktura")
print(f"Zapisano: {catalog}.{schema}.bronze_faktura ✅")
display(df_faktura)
# Kod tworzy DataFrame z listy 'rows', zapisuje go jako tabelę bronze_faktura w wybranym katalogu i schemacie,
# a następnie wyświetla zawartość tej tabeli.